# 07. State Management

The previous notebooks ran one agent on one document at a time and discarded the result
after printing it. In a real pipeline you need to accumulate results across many runs,
detect patterns across documents, and track where each piece of information came from.
That requires state.

## Why state matters in DH
A correspondence archive is not one letter — it is hundreds. The research questions that
matter (Who wrote to whom? Which places appear together? Which names recur across decades?)
can only be answered by comparing results across the whole collection. State is what lets
you do that.

## Concepts
- Accumulating results across multiple agent runs
- Provenance: tracking which document each extracted entity came from
- Cross-document queries on accumulated state
- Co-occurrence analysis
- When in-memory state is enough vs. when to reach for a database

## Dataset
All six documents in `data/` are processed in one pipeline run.

## Colab data setup
If the notebook is opened in Google Colab without cloning the repo, upload `data.zip`
from the repository root using the Files panel, then rerun the setup cell below.

## Further reading
- Agents: https://openai.github.io/openai-agents-python/agents
- Sessions (SDK-level state): https://openai.github.io/openai-agents-python/sessions/
- Tracing: https://openai.github.io/openai-agents-python/tracing/

In [1]:
import os
import sys
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from agents import Agent, Runner, set_tracing_export_api_key, trace

DEFAULT_MODEL = 'gpt-5.4-mini'

In [2]:
candidate_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    zip_candidates = [Path.cwd() / 'data.zip', Path.cwd().parent / 'data.zip']
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is not None:
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(zip_path.parent)
        data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    if 'google.colab' in sys.modules:
        print('Colab detected, but data/ is still missing.')
        print('Upload data.zip from the repository root into the Files panel, then rerun this cell.')
    else:
        raise FileNotFoundError('data/ folder not found.')

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment.'
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])

DOCUMENTS: list[dict[str, str]] = []
for path in sorted(data_dir.glob('*.txt')):
    DOCUMENTS.append({
        'filename': path.name,
        'text': path.read_text(encoding='utf-8').strip(),
    })

print(f'Loaded {len(DOCUMENTS)} documents:')
for doc in DOCUMENTS:
    print(' -', doc['filename'])

Loaded 6 documents:
 - letter_01_madrid_1871.txt
 - letter_02_seville_1894.txt
 - letter_03_ocr_fragment.txt
 - letter_04_barcelona_1902.txt
 - letter_05_porto_1888.txt
 - letter_06_ocr_table.txt


## Step 1: Define the shared extraction schema

We reuse the `Extraction` dataclass from notebook 01. Each run of the agent
produces one `Extraction` object. The challenge is to accumulate these across
all documents while remembering which document each entity came from.

In [3]:
@dataclass
class Extraction:
    people:   list[str] = field(default_factory=list)
    places:   list[str] = field(default_factory=list)
    dates:    list[str] = field(default_factory=list)
    evidence: list[str] = field(default_factory=list)

## Step 2: Define the ExtractionRegister

An `ExtractionRegister` is a plain Python object that accumulates extractions
across runs. It stores every entity alongside its source document, enabling
cross-document queries later.

**Design principle:** keep the register dumb. It stores data; it does not decide
what to extract. The agent makes extraction decisions; the register just persists them.
Separating concerns makes both easier to test.

In [4]:
@dataclass
class EntityRecord:
    value:    str
    category: str        # 'person', 'place', or 'date'
    source:   str        # filename
    evidence: list[str]  # supporting quotes from the same document


class ExtractionRegister:
    """Accumulates EntityRecords across multiple agent runs."""

    def __init__(self):
        self._records: list[EntityRecord] = []

    def add(self, extraction: Extraction, source: str) -> None:
        """Add all entities from one Extraction, tagging them with their source document."""
        for person in extraction.people:
            self._records.append(EntityRecord(person, 'person', source, extraction.evidence))
        for place in extraction.places:
            self._records.append(EntityRecord(place, 'place', source, extraction.evidence))
        for date in extraction.dates:
            self._records.append(EntityRecord(date, 'date', source, extraction.evidence))

    def to_dataframe(self) -> pd.DataFrame:
        """Return all records as a DataFrame for inspection."""
        return pd.DataFrame([
            {'value': r.value, 'category': r.category, 'source': r.source}
            for r in self._records
        ])

    def sources_for(self, value: str) -> list[str]:
        """Return the list of source documents that mention a given entity value."""
        needle = value.strip().lower()
        return sorted({r.source for r in self._records if r.value.strip().lower() == needle})

    def entities_in(self, source: str, category: str | None = None) -> list[str]:
        """Return all entities extracted from a given document, optionally filtered by category."""
        return [
            r.value for r in self._records
            if r.source == source and (category is None or r.category == category)
        ]

    def recurring(self, category: str, min_sources: int = 2) -> dict[str, list[str]]:
        """Return entities of the given category that appear in at least min_sources documents."""
        by_value: dict[str, set[str]] = defaultdict(set)
        for r in self._records:
            if r.category == category:
                by_value[r.value.strip().lower()].add(r.source)
        return {
            value: sorted(sources)
            for value, sources in by_value.items()
            if len(sources) >= min_sources
        }


register = ExtractionRegister()
print('ExtractionRegister ready')

ExtractionRegister ready


## Step 3: Run the agent across all documents

Each document is processed independently. After each run, the extraction is
added to the shared register. The agent itself has no memory of previous runs —
state lives in the register, not in the model.

This is the key architectural point: **the agent is stateless; the register is stateful.**
That separation makes it easy to rerun a single document, swap out the agent, or
replay the pipeline on a different collection.

In [5]:
archive_agent = Agent(
    name='Archive Extractor',
    instructions=(
        'Extract people, places, dates, and evidence from the historical text. '
        'Be conservative: only extract what the text explicitly states. '
        'Include a short direct quote as evidence for each extracted item.'
    ),
    output_type=Extraction,
    model=DEFAULT_MODEL,
)

with trace('state management: full collection run'):
    for doc in DOCUMENTS:
        result = await Runner.run(archive_agent, doc['text'])
        register.add(result.final_output, source=doc['filename'])
        print(f"✓ {doc['filename']} — "
              f"{len(result.final_output.people)} people, "
              f"{len(result.final_output.places)} places, "
              f"{len(result.final_output.dates)} dates")

print(f'\nTotal records in register: {len(register._records)}')

✓ letter_01_madrid_1871.txt — 1 people, 2 places, 2 dates
✓ letter_02_seville_1894.txt — 2 people, 1 places, 1 dates
✓ letter_03_ocr_fragment.txt — 2 people, 1 places, 1 dates
✓ letter_04_barcelona_1902.txt — 5 people, 3 places, 5 dates
✓ letter_05_porto_1888.txt — 4 people, 5 places, 4 dates
✓ letter_06_ocr_table.txt — 8 people, 8 places, 1 dates

Total records in register: 56


## Step 4: Inspect the register

Now that all documents are processed, we can query the register as a whole.
No further model calls are needed — these are plain Python lookups on accumulated state.

In [6]:
# Full table of all extracted entities with provenance.
df = register.to_dataframe()
print(f'Total entities: {len(df)}')
df.sort_values(['category', 'source']).reset_index(drop=True)

Total entities: 56


,value,category,source
0,4 April 1871,date,letter_01_madrid_1871.txt
1,1871,date,letter_01_madrid_1871.txt
2,18 June 1894,date,letter_02_seville_1894.txt
3,1871,date,letter_03_ocr_fragment.txt
4,12 March 1902,date,letter_04_barcelona_1902.txt
5,last autumn,date,letter_04_barcelona_1902.txt
6,Tuesday,date,letter_04_barcelona_1902.txt
7,between 1878 and 1881,date,letter_04_barcelona_1902.txt
8,until the end of April,date,letter_04_barcelona_1902.txt
9,3 September 1888,date,letter_05_porto_1888.txt


In [7]:
# Which places appear in more than one document?
recurring_places = register.recurring('place', min_sources=2)

if recurring_places:
    print('Places mentioned in multiple documents:')
    for place, sources in recurring_places.items():
        print(f'  {place!r:20s} → {sources}')
else:
    print('No place appears in more than one document in this collection.')

No place appears in more than one document in this collection.


In [8]:
# Which people appear in more than one document?
recurring_people = register.recurring('person', min_sources=2)

if recurring_people:
    print('People mentioned in multiple documents:')
    for person, sources in recurring_people.items():
        print(f'  {person!r:30s} → {sources}')
else:
    print('No person appears in more than one document in this collection.')

No person appears in more than one document in this collection.


In [9]:
# Inspect a single document's extracted entities.
target = 'letter_04_barcelona_1902.txt'
print(f'Entities in {target}:')
print('  people:', register.entities_in(target, 'person'))
print('  places:', register.entities_in(target, 'place'))
print('  dates: ', register.entities_in(target, 'date'))

Entities in letter_04_barcelona_1902.txt:
  people: ['Señor Vidal', 'Miquel Font', 'Ramon Vidal', 'Eduard Domènech', 'A. Castelló']
  places: ['Barcelona', 'Biblioteca de Catalunya', 'Girona']
  dates:  ['12 March 1902', 'last autumn', 'Tuesday', 'between 1878 and 1881', 'until the end of April']


In [10]:
# Provenance query: which documents mention Barcelona?
sources = register.sources_for('Barcelona')
print('Documents mentioning Barcelona:', sources)

Documents mentioning Barcelona: ['letter_04_barcelona_1902.txt']


## Step 5: Co-occurrence analysis

A co-occurrence table shows which entities appear together in the same document.
In DH this is a basic building block for correspondence network analysis:
two people who appear in the same letter are likely connected.

In [11]:
# For each document, build a set of (person, place) pairs.
rows = []
for doc in DOCUMENTS:
    people = register.entities_in(doc['filename'], 'person')
    places = register.entities_in(doc['filename'], 'place')
    for person in people:
        for place in places:
            rows.append({
                'person': person,
                'place':  place,
                'source': doc['filename'],
            })

cooccurrence = pd.DataFrame(rows)
print(f'{len(cooccurrence)} person–place co-occurrences across the collection:')
cooccurrence

105 person–place co-occurrences across the collection:


,person,place,source
0,Maria Gomez,Madrid,letter_01_madrid_1871.txt
1,Maria Gomez,Valencia,letter_01_madrid_1871.txt
2,Antonio Ruiz,Seville,letter_02_seville_1894.txt
3,Editor,Seville,letter_02_seville_1894.txt
4,Ma^ria Go'mez,Madrld,letter_03_ocr_fragment.txt
...,...,...,...
100,S4nchez,Gir0na,letter_06_ocr_table.txt
101,S4nchez,Sevi1la,letter_06_ocr_table.txt
102,S4nchez,C4diz,letter_06_ocr_table.txt
103,S4nchez,B4rcelona,letter_06_ocr_table.txt


## Discussion: when is in-memory state enough?

The `ExtractionRegister` above is a Python object that lives in notebook memory.
It disappears when the kernel restarts. That is fine for:

- **Reproducible pipelines** — rerunning the notebook top-to-bottom restores the full state.
- **Small collections** — a few hundred documents fit easily in memory.
- **Exploration** — when you are still deciding what to store and how.

Reach for persistent storage when:

| Signal | Solution |
|--------|----------|
| Collection is too large to reprocess each session | Write to SQLite or a CSV after each run |
| Multiple people or processes share the results | Use a database with concurrent access |
| You need to resume a failed run midway | Checkpoint each document's result to disk |
| Results feed downstream tools (search index, map) | Export to a structured format (JSON Lines, Parquet) |

### SDK-level session state

The OpenAI Agents SDK has a [Sessions](https://openai.github.io/openai-agents-python/sessions/)
concept for sharing conversation history across multiple `Runner.run()` calls to the *same* agent.
That is different from what we built here: sessions preserve the model's conversation context
(turn history), while the `ExtractionRegister` accumulates structured *results* across
independent runs. Both have a role in a production pipeline:

- Use **sessions** when the agent needs memory of previous turns to do its job
  (e.g., a conversational archivist that remembers what was asked before).
- Use a **result store** (like `ExtractionRegister`) when you need to aggregate
  structured outputs for analysis.

## Exercise

The `ExtractionRegister` stores entities but not the relationships between them
within a single document.

Extend `EntityRecord` with an optional `co_entities` field (a list of other entity
values extracted from the same document). Update `ExtractionRegister.add()` to
populate it. Then write a method `co_entities_of(value, category)` that returns
all entities that co-occur with a given value across the collection.

Test it by asking: *Which places co-occur with 'Barcelona' in the same document?*

## Solution hint

In `add()`, collect all entity values from the extraction first, then for each
new `EntityRecord` set `co_entities` to the list of all *other* values from the same run.